# Shared notebook conversation harness
Default: deterministic, zero paid calls. Complete initialization first. Verified notebook and Teams identity mappings can point to the same canonical principal and owned session. Read README.md and AZURE_SETUP.md first. Never paste keys into a notebook.


In [ ]:
# Databricks runs Git-folder notebooks from this directory.
# If your notebook host uses another working directory, set FORECAST_PROJECT_DIR.
import os, sys
from pathlib import Path

ROOT = Path(os.getenv("FORECAST_PROJECT_DIR", str(Path.cwd()))).resolve()
if not (ROOT / "app.py").exists():
    raise ValueError("Set FORECAST_PROJECT_DIR to forecast-intelligence-rca-agent-2.")
sys.path.insert(0, str(ROOT))

In [ ]:
# Identity comes from the authenticated Databricks runtime, not a chat field.
from channels.teams import IdentityMap

login = spark.sql("SELECT current_user() AS login").first()["login"]
identities = IdentityMap(os.environ["IDENTITY_MAP_FILE"])
principal, session_id = identities.resolve("databricks", login)
from runtime import create_core
from agents.forecast_rca_agent import ForecastIntelligenceAgent

# Delta state and its shared paid-call ledger survive notebook/kernel restarts.
core = create_core("spark", "delta", spark=spark)
agent = ForecastIntelligenceAgent(core, principal, session_id)

In [ ]:
# These calls are deterministic unless ENABLE_LLM=true was explicitly configured.
first = agent.run("Analyze forecast performance for P0001 in North.")
print(first.response)
print(agent.run("What about inventory?").response)

To opt into compact evidence ranking, supply Azure settings through secure environment/secret bindings and set ENABLE_LLM=true before creating the core. At most one model request is made per answered turn, with SDK retries disabled, a conservative 1500-byte input ceiling and 150 output tokens. The shared ledger cannot exceed 70 reservations. Never reset it to work around the limit. Genie validation is separate and off by default.
